We are using Google Earth Engine to extract all the sea surface temperature data we need.


In [ ]:
# import library
import ee
import geemap
import geopandas as gpd
import pandas as pd

# authenticate & initilialize earth engine
ee.Authenticate()
ee.Initialize(project='ee-data-viz-challenge-2026')

In [3]:
# get the boundary for an office
# load the shapefile
office = gpd.read_file('/content/drive/MyDrive/BIENPESCA/shapefiles/office_points_latest.shp')

office.columns = ["state_id",  "state",    "office_id",  "office",   "locality",  "cvegeo",   "status",   "stat_abr",  "mun_id",
                          "local_id",  "climate",  "latitud", "longitud",  "altitud",  "letter_id",  "population",  "male_pop",  "feml_pop",
                          "opccupied_households",  "obs_id",   "municipio",  "geometry"]


# https://developers.google.com/earth-engine/datasets/catalog/NOAA_CDR_OISST_V2_1#bands
# 27830 meters pixel size
# 0.01 scale
# measurement in Celsius
# get the image collection (filter for date)
sst_collection = (
    ee.ImageCollection('NOAA/CDR/OISST/V2_1')
    .filterDate('2006-01-01', '2024-12-31')
    .select('sst')
)


In [4]:
# Convert office into ee.FeatureCollection with 100 km buffe
def create_buffered_feature(row):
    coords = row.geometry.coords[0]
    ee_point = ee.Geometry.Point(coords)
    #buffer = ee_point.buffer(100000)  # 100 km
    buffer = ee_point.buffer(200000)  # 200 km
    return ee.Feature(buffer).set({'office_id': row.office_id})

# Apply the conversion
features = [create_buffered_feature(row) for idx, row in office.iterrows()]
fc = ee.FeatureCollection(features)

In [7]:
# Define SST extraction function
def extract_monthly_stats(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    mean_img = sst_collection.filterDate(start, end).mean()

    def extract_for_feature(f):
        mean = mean_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=f.geometry(),
            scale=1000,
            maxPixels=1e8
        )
        std = mean_img.reduceRegion(
            reducer=ee.Reducer.stdDev(),
            geometry=f.geometry(),
            scale=1000,
            maxPixels=1e8
        )
        return f.set({
            'year': year,
            'month': month,
            'avg_sst': mean.get('sst'),
            'sd_sst': std.get('sst')
        })

    return fc.map(extract_for_feature)

In [8]:
#  Aggregate over all months
years = list(range(2006, 2025))
months = list(range(1, 13))

results = []
for y in years:
    for m in months:
        results.append(extract_monthly_stats(y, m))

# Merge all feature collections into one
all_months_fc = ee.FeatureCollection(results).flatten()

In [9]:
# Export as CSV to Google Drive
import time

task = ee.batch.Export.table.toDrive(
    collection=all_months_fc,
    description='monthly_sst_export',
    folder='BIENPESCA_exports',
    fileNamePrefix='sst_all_offices_2006_2024_200km_buffer',
    fileFormat='CSV'
)
task.start()


while task.active():
    print("Exporting... status:", task.status()['state'])
    time.sleep(30)

print("Export complete.")

Exporting... status: READY
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... sta

IsADirectoryError: [Errno 21] Is a directory: '/content/drive/MyDrive/BIENPESCA/sst/'